# Olist Marketplace Intelligence — Customer & Sales Analysis

**Business question:** Olist is a Brazilian e-commerce marketplace with
strong order volume, but a strikingly low repeat-purchase rate. This
notebook investigates *why*, tests four competing hypotheses with
controlled SQL queries, builds a customer-value segmentation, and trains a
predictive model to see whether the drivers found in SQL hold up under a
combined, cross-validated test.

**Dataset:** [Olist Brazilian E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
(Kaggle) — ~99,000 orders across 9 relational tables (customers, orders,
order items, payments, reviews, products, sellers, geolocation, category
translations).

**Structure of this notebook:**
1. Setup & data loading
2. Data quality checks
3. Executive KPIs & trend analysis (SQL)
4. Regional & category performance (SQL)
5. Customer behavior deep-dive: the 97% one-time-buyer question
6. Hypothesis testing — delivery, category, payment, geography (SQL)
7. Predictive modeling — feature engineering, model comparison, final model
8. RFM customer segmentation
9. Exporting outputs for Power BI

Full write-up of findings and recommendations: `docs/data_cleaning_log.md`.
A log of real technical issues hit while building this (and how each was
diagnosed/fixed) is in `docs/issues_and_debugging.md`.


## 1. Setup & Data Loading

Loads the 9 raw Olist CSVs into a local SQLite database. **Important:**
this must run against local disk, not a Google-Drive-mounted path — SQLite
needs fast random read/write for indexing, which a network-mounted drive
cannot provide. (This caused a real 10+ minute query hang during
development; see `docs/issues_and_debugging.md`, issue #1.)


In [ ]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("olist.db")  # local disk — required for index performance

files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table_name, filename in files.items():
    df = pd.read_csv(filename)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"{table_name}: {len(df)} rows loaded")


Indexes on every join/filter column used downstream. Without these,
the multi-table hypothesis-testing queries in Section 6 take 10+ minutes
instead of seconds — a real bottleneck hit during development once the
database was moved off Google Drive onto local disk.

In [ ]:
%%time
cur = conn.cursor()
indexes = [
    "CREATE INDEX IF NOT EXISTS idx_orders_customer ON orders(customer_id)",
    "CREATE INDEX IF NOT EXISTS idx_orders_status ON orders(order_status)",
    "CREATE INDEX IF NOT EXISTS idx_orders_id ON orders(order_id)",
    "CREATE INDEX IF NOT EXISTS idx_items_order ON order_items(order_id)",
    "CREATE INDEX IF NOT EXISTS idx_items_product ON order_items(product_id)",
    "CREATE INDEX IF NOT EXISTS idx_customers_id ON customers(customer_id)",
    "CREATE INDEX IF NOT EXISTS idx_customers_unique ON customers(customer_unique_id)",
    "CREATE INDEX IF NOT EXISTS idx_products_id ON products(product_id)",
    "CREATE INDEX IF NOT EXISTS idx_products_cat ON products(product_category_name)",
    "CREATE INDEX IF NOT EXISTS idx_cattrans ON category_translation(product_category_name)",
    "CREATE INDEX IF NOT EXISTS idx_payments_order ON order_payments(order_id)",
    "CREATE INDEX IF NOT EXISTS idx_reviews_order ON order_reviews(order_id)",
]
for idx in indexes:
    cur.execute(idx)
conn.commit()
print("Indexes created.")


## 2. Data Quality Checks

Before any analysis, we need to understand: what's missing, what's the
usable date range, and which order statuses actually represent completed
sales. These checks directly determine the filtering logic used in every
query that follows.

In [ ]:
pd.read_sql("SELECT COUNT(*) FROM orders WHERE order_delivered_customer_date IS NULL", conn)

In [ ]:
pd.read_sql("SELECT MIN(order_purchase_timestamp), MAX(order_purchase_timestamp) FROM orders", conn)

In [ ]:
pd.read_sql("SELECT order_status, COUNT(*) FROM orders GROUP BY order_status", conn)

In [ ]:
# Duplicate order_id check
pd.read_sql("SELECT order_id, COUNT(*) c FROM orders GROUP BY order_id HAVING c > 1", conn)

In [ ]:
# Monthly order volume — checking for partial-month artifacts at the edges of the date range
pd.read_sql("""
SELECT strftime('%Y-%m', order_purchase_timestamp) AS month, COUNT(*)
FROM orders GROUP BY month ORDER BY month
""", conn)

**Findings from this section (full detail in `docs/data_cleaning_log.md`):**
- 99,441 orders total; 97% delivered, 625 canceled, 609 unavailable —
  **only `delivered` orders are used for revenue calculations** going forward.
- No duplicate `order_id` values.
- 2,965 missing delivery dates, consistent with non-delivered statuses (not an error).
- **Usable date range: 2017-01 to 2018-08.** 2016-09/2016-12 (329 orders) and
  2018-09/2018-10 (20 orders) are partial-month artifacts of the data
  collection window, not real business activity — excluded from all
  analysis to avoid misleading edge effects on trend charts.

**A second, more important data-quality catch:** `customers.customer_id`
is unique *per order*, not per person — a returning customer gets a new
`customer_id` on every order. The real person-level key is
`customer_unique_id`. The query below demonstrates the problem directly.

In [ ]:
# customer_id vs customer_unique_id — proving the difference matters
pd.read_sql("""
SELECT COUNT(DISTINCT customer_id) AS order_level_ids,
       COUNT(DISTINCT customer_unique_id) AS person_level_ids
FROM customers
""", conn)
# order_level_ids > person_level_ids confirms customer_id inflates the true
# customer count. Every customer-level query from here on uses
# customer_unique_id.

## 3. Executive KPIs & Monthly Trend

Baseline business metrics, scoped to `delivered` orders and the 2017-01 to
2018-08 date range established above. Uses `customer_unique_id` throughout.

In [ ]:
query = """
SELECT
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT c.customer_unique_id) AS total_customers,
    ROUND(SUM(oi.price), 2) AS total_revenue,
    ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value,
    ROUND(SUM(oi.freight_value), 2) AS total_freight
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp < '2018-09-01'
"""
pd.read_sql(query, conn)
# Result: 96,211 orders | 93,104 customers | R$13,181,027.13 revenue
#         R$137.00 AOV | R$2,192,092.88 freight (~16.6% of revenue)

**Revenue definition:** product price only (`order_items.price`).
Freight is tracked separately rather than folded into "revenue," since it's
a pass-through shipping cost, not a reflection of sales performance.

### Monthly revenue trend
Uses the `LAG()` window function to compute month-over-month growth.

In [ ]:
query = """
WITH monthly AS (
    SELECT
        strftime('%Y-%m', o.order_purchase_timestamp) AS order_month,
        SUM(oi.price) AS revenue,
        COUNT(DISTINCT o.order_id) AS orders
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY order_month
)
SELECT
    order_month,
    ROUND(revenue, 2) AS revenue,
    orders,
    ROUND(revenue - LAG(revenue) OVER (ORDER BY order_month), 2) AS mom_change,
    ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY order_month))
          / LAG(revenue) OVER (ORDER BY order_month), 1) AS mom_growth_pct
FROM monthly
ORDER BY order_month
"""
pd.read_sql(query, conn)
# Finding: steady growth through 2017 (peaking +52% MoM in Nov 2017, likely
# Black Friday), but growth stalled in 2018 — May 2018 (R$977K) was the
# effective peak; Jun-Aug 2018 stayed flat/below it. Deceleration, not decline.

## 4. Regional & Category Performance

### Revenue by state

In [ ]:
query = """
SELECT
    c.customer_state,
    COUNT(DISTINCT c.customer_unique_id) AS customers,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(SUM(oi.price), 2) AS revenue,
    ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN customers c ON c.customer_id = o.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp < '2018-09-01'
GROUP BY c.customer_state
ORDER BY revenue DESC
"""
pd.read_sql(query, conn)
# Finding: Sao Paulo (SP) = 42% of customers, 38% of revenue — more than the
# next 5 states combined. But SP has one of the LOWEST average order values
# (R$125), while small/remote states (PB, AP, AC) have the highest AOV
# (~R$200+). Remote customers order less often but spend more per order.

### Category performance and top products
The second query demonstrates `RANK() OVER (PARTITION BY ...)` to find the
top 3 products *within* each category, not just overall.

In [ ]:
query = """
SELECT
    t.product_category_name_english AS category,
    ROUND(SUM(oi.price), 2) AS revenue,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(SUM(oi.price) / COUNT(DISTINCT o.order_id), 2) AS avg_order_value
FROM orders o
JOIN order_items oi ON oi.order_id = o.order_id
JOIN products p ON p.product_id = oi.product_id
JOIN category_translation t ON t.product_category_name = p.product_category_name
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp < '2018-09-01'
GROUP BY category
ORDER BY revenue DESC
LIMIT 15
"""
pd.read_sql(query, conn)
# Finding: no single category dominates (top category health_beauty is only
# ~9% of total) — concentration risk is geographic, not product-based.

In [ ]:
query = """
WITH product_revenue AS (
    SELECT
        t.product_category_name_english AS category,
        oi.product_id,
        ROUND(SUM(oi.price), 2) AS revenue,
        RANK() OVER (PARTITION BY t.product_category_name_english
                      ORDER BY SUM(oi.price) DESC) AS rank_in_category
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id
    JOIN category_translation t ON t.product_category_name = p.product_category_name
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY category, oi.product_id
)
SELECT * FROM product_revenue
WHERE rank_in_category <= 3
ORDER BY revenue DESC
LIMIT 30
"""
pd.read_sql(query, conn)

## 5. Customer Behavior Deep-Dive: Why Don't Customers Return?

### 5.1 Building the RFM base table

Recency (days since last order), Frequency (order count), Monetary (total
spend) per customer — the foundation for both the segmentation in Section
8 and the hypothesis testing below.

In [ ]:
query = """
WITH snapshot AS (SELECT '2018-09-01' AS snapshot_date),
customer_orders AS (
    SELECT
        c.customer_unique_id,
        MAX(o.order_purchase_timestamp) AS last_order_date,
        COUNT(DISTINCT o.order_id) AS frequency,
        SUM(oi.price) AS monetary
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY c.customer_unique_id
)
SELECT
    customer_unique_id,
    CAST(JULIANDAY((SELECT snapshot_date FROM snapshot)) - JULIANDAY(last_order_date) AS INT) AS recency_days,
    frequency,
    ROUND(monetary, 2) AS monetary
FROM customer_orders
"""
rfm = pd.read_sql(query, conn)
rfm.head(10)

In [ ]:
rfm[['recency_days', 'frequency', 'monetary']].describe()
# Median AND 75th-percentile frequency = 1 — at least 75% of customers are
# one-time buyers. This rules out standard 5-way RFM quintile scoring on
# Frequency (no spread to split); segmentation in Section 8 uses Recency +
# Monetary only.

### 5.2 Confirming the exact scale of the pattern

In [ ]:
query = """
WITH customer_orders AS (
    SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS frequency
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY c.customer_unique_id
)
SELECT
    SUM(CASE WHEN frequency = 1 THEN 1 ELSE 0 END) AS one_time_customers,
    SUM(CASE WHEN frequency > 1 THEN 1 ELSE 0 END) AS repeat_customers,
    ROUND(100.0 * SUM(CASE WHEN frequency = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_one_time
FROM customer_orders
"""
pd.read_sql(query, conn)
# Result: 90,315 one-time (97.0%) vs 2,789 repeat customers.
# This became the project's central question, investigated in Section 6.

## 6. Hypothesis Testing: What Predicts Repeat Purchase?

Four hypotheses are tested below, each isolating one candidate driver. All
four queries control for **reorder-eligibility**: only customers whose
first order was before 2018-03-01 are included, guaranteeing everyone had
6+ months to place a second order before the 2018-09-01 data cutoff.
Without this control, recently-acquired customers would be misclassified
as "one-time" simply for lack of time, biasing every comparison.

### 6.1 Hypothesis: Delivery experience predicts repeat purchase

In [ ]:
query = """
WITH first_orders AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id
                            ORDER BY o.order_purchase_timestamp) AS rn
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-03-01'
),
first_only AS (SELECT * FROM first_orders WHERE rn = 1),
customer_freq AS (
    SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS frequency
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY c.customer_unique_id
)
SELECT
    CASE WHEN cf.frequency = 1 THEN 'One-time' ELSE 'Repeat' END AS customer_type,
    COUNT(*) AS customers,
    ROUND(AVG(r.review_score), 2) AS avg_review_score,
    ROUND(AVG(JULIANDAY(fo.order_delivered_customer_date) - JULIANDAY(fo.order_purchase_timestamp)), 1) AS avg_delivery_days,
    ROUND(AVG(JULIANDAY(fo.order_delivered_customer_date) - JULIANDAY(fo.order_estimated_delivery_date)), 1) AS avg_days_early_or_late
FROM first_only fo
JOIN customer_freq cf ON cf.customer_unique_id = fo.customer_unique_id
LEFT JOIN order_reviews r ON r.order_id = fo.order_id
GROUP BY customer_type
"""
pd.read_sql(query, conn)
# RESULT: NO effect. One-time (4.13) vs repeat (4.19) review scores nearly
# identical; both delivered ~13 days, both ~11-12 days early vs estimate.
# Hypothesis ruled out.

### 6.2 Hypothesis: Product category (first purchase) predicts repeat purchase

In [ ]:
query = """
WITH first_orders AS (
    SELECT
        c.customer_unique_id,
        oi.product_id,
        ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id
                            ORDER BY o.order_purchase_timestamp) AS rn
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-03-01'
),
first_only AS (SELECT * FROM first_orders WHERE rn = 1),
customer_freq AS (
    SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS frequency
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY c.customer_unique_id
)
SELECT
    t.product_category_name_english AS category,
    SUM(CASE WHEN cf.frequency = 1 THEN 1 ELSE 0 END) AS one_time,
    SUM(CASE WHEN cf.frequency > 1 THEN 1 ELSE 0 END) AS repeat,
    ROUND(100.0 * SUM(CASE WHEN cf.frequency > 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS repeat_rate_pct,
    COUNT(*) AS total_first_time_buyers
FROM first_only fo
JOIN products p ON p.product_id = fo.product_id
JOIN category_translation t ON t.product_category_name = p.product_category_name
JOIN customer_freq cf ON cf.customer_unique_id = fo.customer_unique_id
GROUP BY category
HAVING total_first_time_buyers >= 200
ORDER BY repeat_rate_pct DESC
"""
pd.read_sql(query, conn)
# RESULT: NO clear pattern. Repeat rate compressed into a narrow 2.0%-9.1%
# band across every major category. Durable goods (furniture, appliances)
# did NOT show lower repeat rates than consumables, contrary to hypothesis.
# Hypothesis ruled out.

### 6.3 Hypothesis: Payment behavior predicts repeat purchase

In [ ]:
query = """
WITH first_orders AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id
                            ORDER BY o.order_purchase_timestamp) AS rn
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-03-01'
),
first_only AS (SELECT * FROM first_orders WHERE rn = 1),
customer_freq AS (
    SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS frequency
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY c.customer_unique_id
)
SELECT
    CASE WHEN cf.frequency = 1 THEN 'One-time' ELSE 'Repeat' END AS customer_type,
    ROUND(AVG(pay.payment_installments), 2) AS avg_installments,
    ROUND(AVG(pay.payment_value), 2) AS avg_payment_value
FROM first_only fo
JOIN customer_freq cf ON cf.customer_unique_id = fo.customer_unique_id
JOIN order_payments pay ON pay.order_id = fo.order_id
GROUP BY customer_type
"""
pd.read_sql(query, conn)
# RESULT: WEAK effect. Repeat customers spend slightly LESS per order
# (R$134 vs R$151) and use marginally more installments (3.24 vs 2.88).
# The strongest signal of the four — later confirmed as the #1 feature in
# the churn model (Section 7).

### 6.4 Hypothesis: Geography (customer state) predicts repeat purchase

In [ ]:
query = """
WITH first_orders AS (
    SELECT c.customer_unique_id, c.customer_state,
           ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id
                               ORDER BY o.order_purchase_timestamp) AS rn
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-03-01'
),
first_only AS (SELECT * FROM first_orders WHERE rn = 1),
customer_freq AS (
    SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS frequency
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY c.customer_unique_id
)
SELECT
    fo.customer_state,
    COUNT(*) AS customers,
    ROUND(100.0 * SUM(CASE WHEN cf.frequency > 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS repeat_rate_pct
FROM first_only fo
JOIN customer_freq cf ON cf.customer_unique_id = fo.customer_unique_id
GROUP BY fo.customer_state
HAVING customers >= 300
ORDER BY repeat_rate_pct DESC
"""
pd.read_sql(query, conn)
# RESULT: WEAK effect. Repeat rate ranges 2.0% (CE) to 4.3% (SP/MT) — ~2x
# spread, but even the best state stays under 5%.

### 6.5 Conclusion from hypothesis testing

| Hypothesis | Result |
|---|---|
| Delivery experience | No effect |
| Product category | No clear pattern |
| Payment behavior | Weak effect |
| Geography | Weak effect |

**No single lever explains the 97% one-time-purchase rate.** This is a
structural, platform-wide characteristic — not a fixable operational
failure isolated to one segment, category, or region. Section 7 tests
whether *combining* all these weak signals into one model does better than
any single factor alone.

## 7. Predictive Modeling: Combining All Signals

### 7.1 Feature extraction

One row per customer (first order only, same reorder-eligibility window as
Section 6), with every signal tested individually above — category,
price, installments, review score, delivery timing, state — combined into
one feature set.

In [ ]:
query = """
WITH first_orders AS (
    SELECT
        c.customer_unique_id,
        c.customer_state,
        o.order_id,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        ROW_NUMBER() OVER (PARTITION BY c.customer_unique_id ORDER BY o.order_purchase_timestamp) AS rn
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-03-01'
),
first_only AS (SELECT * FROM first_orders WHERE rn = 1),
customer_freq AS (
    SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS frequency
    FROM orders o
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp >= '2017-01-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY c.customer_unique_id
),
order_category AS (
    -- a customer's first order can contain multiple items/categories; take
    -- the category of their single highest-value item as "the" category
    SELECT fo.customer_unique_id, t.product_category_name_english AS category,
           ROW_NUMBER() OVER (PARTITION BY fo.customer_unique_id ORDER BY oi.price DESC) AS item_rank
    FROM first_only fo
    JOIN order_items oi ON oi.order_id = fo.order_id
    JOIN products p ON p.product_id = oi.product_id
    JOIN category_translation t ON t.product_category_name = p.product_category_name
)
SELECT
    fo.customer_unique_id,
    fo.customer_state,
    oc.category,
    cf.frequency,
    CASE WHEN cf.frequency > 1 THEN 1 ELSE 0 END AS repeat_customer,
    pay.payment_installments,
    pay.payment_value,
    pay.payment_type,
    r.review_score,
    ROUND(JULIANDAY(fo.order_delivered_customer_date) - JULIANDAY(fo.order_purchase_timestamp), 1) AS delivery_days,
    ROUND(JULIANDAY(fo.order_delivered_customer_date) - JULIANDAY(fo.order_estimated_delivery_date), 1) AS days_early_or_late,
    CAST(strftime('%w', fo.order_purchase_timestamp) AS INTEGER) AS purchase_weekday
FROM first_only fo
JOIN customer_freq cf ON cf.customer_unique_id = fo.customer_unique_id
JOIN order_category oc ON oc.customer_unique_id = fo.customer_unique_id AND oc.item_rank = 1
LEFT JOIN order_payments pay ON pay.order_id = fo.order_id
LEFT JOIN order_reviews r ON r.order_id = fo.order_id
"""
df = pd.read_sql(query, conn)
print(df.shape)  # (57149, 12)
df.head()

# Note: purchase_weekday is explicitly CAST to INTEGER in SQL (SQLite's
# strftime('%w', ...) returns text by default) — this avoids a dtype error
# that otherwise surfaces later if a stricter downstream library (e.g.
# XGBoost) enforces numeric types, since pandas/scikit-learn silently
# tolerate numeric-looking text but not every library does.

### 7.2 Handling missing values and target imbalance

In [ ]:
df['no_review'] = df['review_score'].isna().astype(int)
df['review_score'] = df['review_score'].fillna(df['review_score'].median())

# delivery_days / days_early_or_late: only 2 nulls each — drop, not worth
# engineering around such a small number of rows
df = df.dropna(subset=['delivery_days', 'days_early_or_late'])

print(df.isna().sum().sum(), "missing values remaining")
print(df['repeat_customer'].value_counts(normalize=True))
# repeat_customer: 0 = 95.6%, 1 = 4.4% — severely imbalanced. Every model
# below uses class_weight='balanced' (or equivalent) and is evaluated on
# ROC-AUC, not accuracy, which would be misleadingly high on this target.

In [ ]:
features_df = pd.get_dummies(
    df.drop(columns=['customer_unique_id', 'frequency', 'repeat_customer']),
    columns=['category', 'customer_state', 'payment_type'],
    drop_first=True
)

X = features_df
y = df['repeat_customer']

print(X.shape)
# frequency is deliberately excluded — it's how repeat_customer was
# derived, so including it would be direct data leakage.

### 7.3 Model comparison (5-fold cross-validated ROC-AUC)

Four models compared: a dummy baseline (sanity check), Logistic
Regression, Random Forest, and Gradient Boosting. 5-fold CV is used
instead of a single train/test split for a more stable estimate, since the
severely imbalanced target (~4.4% positive) makes any single split
sensitive to which examples happen to land in the test fold.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Dummy (baseline)": DummyClassifier(strategy="stratified", random_state=42),
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=6, min_samples_leaf=20,
        class_weight="balanced", random_state=42
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
    ),
}

results = []
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    results.append({"model": name, "mean_roc_auc": scores.mean(), "std": scores.std()})
    print(f"{name}: {scores.mean():.3f} (+/- {scores.std():.3f})")

results_df = pd.DataFrame(results).sort_values("mean_roc_auc", ascending=False)
results_df

# Results:
#   Dummy (baseline):     0.498
#   Logistic Regression:  0.614
#   Random Forest:        0.621
#   Gradient Boosting:    0.634  <- best, used as the final model below

**Note on extensions tested but not adopted:** hyperparameter tuning
via `GridSearchCV` and an XGBoost variant were both tried as follow-ups.
Neither meaningfully improved on the Gradient Boosting result above once
compared on a consistent evaluation setup — full detail (including a
cross-validation fold-count mismatch that initially made tuning look
*worse*, and an XGBoost dtype error from an untyped column) is documented
in `docs/issues_and_debugging.md`, issues #7 and #8. Omitted from this
notebook to keep the modeling narrative focused on what was actually used.

### 7.4 Final model: fit, evaluate, extract feature importance

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

final_model = GradientBoostingClassifier(
    n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
importances = pd.Series(final_model.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(15)
# Top features: payment_value (32%), days_early_or_late (15%),
# delivery_days (7%), payment_installments (7%). review_score does not
# rank in the top 15 — satisfaction alone doesn't explain repeat behavior,
# consistent with the SQL finding in Section 6.1.

### 7.5 Honest reframe: from classifier to targeting tool

The raw classification metrics above are weak (best F1 for the repeat
class: ~0.02–0.11) — not because the model is broken, but because the SQL
investigation already showed there's no strong individual driver to learn.
Rather than force a "prediction" narrative, the model is more honestly
used as a **ranking/targeting tool**: are customers the model scores
highest actually more likely to return, even if no individual prediction
is reliable?

In [ ]:
top_10pct_cutoff = pd.Series(y_proba).quantile(0.90)
predicted_top10 = y_proba >= top_10pct_cutoff

print(f"Customers in top 10% by predicted probability: {predicted_top10.sum()}")
print(f"Of those, actually repeat customers: {y_test[predicted_top10].sum()}")
print(f"Baseline repeat rate: {y_test.mean():.3f}")
print(f"Repeat rate within top 10% flagged: {y_test[predicted_top10].mean():.3f}")
# Result: 9.5% repeat rate in the top decile vs 4.4% baseline — a 2.1x lift.
# Not reliable enough to predict any single customer, but genuinely useful
# for prioritizing a win-back/loyalty campaign budget toward the customers
# most likely to respond.

## 8. RFM Customer Segmentation

Because 75%+ of customers have Frequency = 1 (Section 5.1), standard 5-way
quintile scoring on Frequency isn't viable — there's no spread to split.
Segmentation instead uses **Recency + Monetary only**.

In [ ]:
rfm['r_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5, 4, 3, 2, 1]).astype(int)  # recent = high score
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment(row):
    r, m = row['r_score'], row['m_score']
    if r >= 4 and m >= 4:
        return 'Champions'
    if r >= 4 and m <= 3:
        return 'Recent, Lower Spend'
    if r <= 2 and m >= 4:
        return 'High Value At Risk'
    if r <= 2 and m <= 2:
        return 'Lost / Low Value'
    return 'Needs Attention'

rfm['segment'] = rfm.apply(segment, axis=1)
rfm['segment'].value_counts()

In [ ]:
rfm.groupby('segment')['monetary'].agg(['count', 'mean', 'sum'])
# Champions: 15,404 customers | avg R$270.72 | total R$4,170,128.14
# High Value At Risk: 14,429 | avg R$278.51 | total R$4,018,607.75
# Needs Attention: 26,083 | avg R$121.11 | total R$3,158,956.69
# Recent, Lower Spend: 21,915 | avg R$55.74 | total R$1,221,589.10
# Lost / Low Value: 15,273 | avg R$40.05 | total R$611,745.45
#
# "Champions" + "High Value At Risk" = R$8.19M — 62% of total revenue —
# sitting in customers who are either currently engaged or have already
# gone quiet. Single most important number for the recommendations page.

## 9. Exporting Outputs for Power BI

Saves the RFM segmentation and cleaned source tables as CSVs, which feed
directly into the Power BI data model (see `dashboard/olist_dashboard.pbix`
and the star-schema diagram in the README).

In [ ]:
rfm.to_csv('rfm_segments.csv', index=False)

tables_to_export = ['customers', 'orders', 'order_items', 'products',
                     'category_translation', 'order_payments', 'order_reviews']
for t in tables_to_export:
    pd.read_sql(f"SELECT * FROM {t}", conn).to_csv(f'clean_{t}.csv', index=False)
    print(f"Exported clean_{t}.csv")

print("\nAll outputs saved.")

## Summary

This notebook took a marketplace with strong revenue growth but a 97%
one-time-purchase rate, tested four competing explanations with controlled
SQL queries, ruled out three of them (delivery, category, geography as
weak/no effect) and found one weak lead (payment behavior) — then
confirmed with an independently cross-validated model that no combination
of available signals predicts repeat purchase reliably (ROC-AUC 0.634).
Rather than treat that as a dead end, the model was reframed into a
targeting tool with a genuine 2.1x lift, and paired with an RFM
segmentation identifying R$8.19M in revenue concentrated in high-value
customers. Full findings, recommendations, and a log of real technical
issues resolved during development are in the `docs/` folder of the
project repository.
